In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

In [ ]:
from production_app.data import load_dataset, split_data

In [ ]:
X, y = load_dataset()
X_train, X_test, y_train, y_test = split_data(X, y)

In [ ]:
print("Train:", X_train.shape, " Test:", X_test.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

baseline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000)),
    ]
)

score = cross_val_score(baseline, X_train, y_train, cv=cv).mean()
print("Baseline, 30 features: CV accuracy =", round(score, 3))

In [ ]:
from sklearn.feature_selection import SelectFromModel

for C in [1.0, 0.5, 0.1, 0.05, 0.01]:
    pipe = Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "select",
                SelectFromModel(
                    LogisticRegression(penalty="l1", solver="liblinear", C=C)
                ),
            ),
            ("model", LogisticRegression(max_iter=1000)),
        ]
    )

    score = cross_val_score(pipe, X_train, y_train, cv=cv).mean()

    pipe.fit(X_train, y_train)
    kept = pipe["select"].get_support().sum()

    print(f"C={C}: {kept} features kept, CV accuracy = {score:.3f}")

In [ ]:
from sklearn.metrics import make_scorer, recall_score

recall_malignant = make_scorer(recall_score, pos_label=0)

for name, C in [("baseline (30)", None), ("C=0.1", 0.1), ("C=0.05", 0.05)]:
    steps = [("scaler", StandardScaler())]
    if C is not None:
        steps.append(
            (
                "select",
                SelectFromModel(
                    LogisticRegression(penalty="l1", solver="liblinear", C=C)
                ),
            )
        )
    steps.append(("model", LogisticRegression(max_iter=1000)))
    score = cross_val_score(
        Pipeline(steps), X_train, y_train, cv=cv, scoring=recall_malignant
    ).mean()
    print(f"{name}: CV recall (malignant) = {score:.3f}")

In [ ]:
for C in [1.0, 0.5, 0.3, 0.2, 0.1, 0.05]:
    pipe = Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "select",
                SelectFromModel(
                    LogisticRegression(penalty="l1", solver="liblinear", C=C)
                ),
            ),
            ("model", LogisticRegression(max_iter=1000)),
        ]
    )

    acc = cross_val_score(pipe, X_train, y_train, cv=cv).mean()
    rec = cross_val_score(
        pipe, X_train, y_train, cv=cv, scoring=recall_malignant
    ).mean()

    pipe.fit(X_train, y_train)
    kept = pipe["select"].get_support().sum()

    print(
        f"C={C}: {kept} features, accuracy = {acc:.3f}, recall (malignant) = {rec:.3f}"
    )

In [ ]:
from sklearn.metrics import accuracy_score, recall_score

baseline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000)),
    ]
)
selected = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "select",
            SelectFromModel(
                LogisticRegression(penalty="l1", solver="liblinear", C=0.3)
            ),
        ),
        ("model", LogisticRegression(max_iter=1000)),
    ]
)

for name, pipe in [("Baseline (30)", baseline), ("L1, C=0.3", selected)]:
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    print(
        f"{name}: accuracy = {accuracy_score(y_test, y_pred):.3f}, "
        f"recall (malignant) = {recall_score(y_test, y_pred, pos_label=0):.3f}"
    )

kept = list(X_train.columns[selected["select"].get_support()])
print(f"\n{len(kept)} features kept:")
for f in kept:
    print(" -", f)

## Feature selection: research and decision

| Family | How it works | Example | Strength | Limitation |
|---|---|---|---|---|
| Filter | Scores each feature on its own with a statistic, keeps the top ones | SelectKBest (ANOVA F-test) | Very fast, model-independent | Ignores redundancy: keeps several features that say the same thing |
| Wrapper | Trains the model on feature subsets and keeps what performs best | RFE | Picks features that work well together for this model | Slow: trains the model many times |
| Embedded | Selection happens during training: L1 pushes weak weights to exactly 0 | L1 Logistic Regression + SelectFromModel | One training run, model-aware, handles redundancy | Number of kept features depends on the strength C, not chosen directly |

**Considered but not used:** PCA reduces dimensions but creates new mixed components, so users would still need to enter all 30 values.

**Chosen: Embedded (L1, `solver="liblinear"`).** C was chosen by 5-fold stratified cross-validation on the training set only (selector inside the pipeline, so no leakage):

| C | Features | CV accuracy | CV recall (malignant) |
|---|---|---|---|
| baseline | 30 | 0.978 | 0.965 |
| 1.0 | 16 | 0.963 | 0.924 |
| 0.5 | 12 | 0.965 | 0.935 |
| **0.3** | **11** | **0.974** | **0.953** |
| 0.2 | 10 | 0.967 | 0.947 |
| 0.1 | 8 | 0.965 | 0.941 |
| 0.05 | 7 | 0.965 | 0.947 |

C = 0.3 is the best balance: 30 → 11 inputs with the smallest loss in recall for malignant (the costliest mistake).

**Test set, used once at the end:** baseline accuracy 0.982 / recall 0.976; selected (11 features) accuracy 0.982 / recall 0.976.

**Selected features (dataset order):** mean texture, mean concave points, radius error, fractal dimension error, worst radius, worst texture, worst area, worst smoothness, worst concavity, worst concave points, worst symmetry.